# 03 Random forest baseline: RGB versus 13 bands

**Purpose.** Train the classical baseline that the deep models must beat: a random forest on simple statistics of each band, once with the RGB bands and once with all 13 bands. The random forest is one of the standard scikit-learn models of the course (lecture_2_introduction_to_ML.pdf, slide 17), and building a baseline before the more complex models is step 6 of the course workflow for supervised learning (lecture_2_introduction_to_ML.pdf, slide 46).

**Functions.** `src/piece_of_crop/features.py` (`extract_rf_features`, `tune_random_forest`), `data.py` (`read_split`, `get_cache`) and `train.py` (`predictions_table`, the shared format for saved predictions).

**Reads.** `results/split.csv`, `results/band_stats.csv` and the patch cache `data/eurosat_cache.npy` (built from the GeoTIFFs on first use).

**Saves.** `results/rf_tuning.csv`, `results/rf_results.csv`, six test prediction files `results/predictions/rf_<input>_seed<n>.csv`, the figures `rf_comparison.png` and `rf_feature_importance_13band.png` in `results/figures/`, and the trained models in `models/` (not in Git; notebook 07 uses them).

## 1. Setup and shared data from notebook 02

All functions come from the installed `piece_of_crop` package. `find_project_root` finds the project folder whether the notebook runs from the project root or from `notebooks/`.

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score

from piece_of_crop import (CLASS_NAMES, extract_rf_features, find_project_root, get_cache, predictions_table,
                           read_split, tune_random_forest)

SEEDS = (0, 1, 2)
N_ESTIMATORS = 200

project_root = find_project_root()
data_dir = project_root / "data" / "EuroSAT_MS"
results_dir = project_root / "results"
figures_dir = results_dir / "figures"
predictions_dir = results_dir / "predictions"
models_dir = project_root / "models"
for folder in (results_dir, figures_dir, predictions_dir, models_dir):
    folder.mkdir(parents=True, exist_ok=True)

split = read_split(results_dir / "split.csv")
band_stats = pd.read_csv(results_dir / "band_stats.csv")
cache = get_cache(split, data_dir, project_root / "data" / "eurosat_cache.npy")

print("Project root:", project_root)
print("Patches per split:", split["split"].value_counts().to_dict())
print("Cache shape:", cache.shape)

## 2. Features: mean and standard deviation of each band

A random forest needs one row of numbers per patch, not an image. The course notebook on image classification with classical models does exactly this: after a random forest on the raw pixels, it turns each image into features by "computing the mean and the standard deviation of the different colour values" (ImageClassificationML.pdf, page 5), and lists the mean and standard deviation of the pixel values as basic statistical features (page 11). We use the same two statistics for every band:

- **RGB:** mean and standard deviation of B02, B03 and B04, which gives 6 features.
- **13 bands:** mean and standard deviation of all 13 bands, plus the mean and standard deviation of NDVI over the patch, which gives 28 features. NDVI = (B08 - B04) / (B08 + B04) compares near-infrared and red light, which healthy plants reflect very differently.

**The asymmetry is deliberate.** NDVI needs the near-infrared band B08, which an RGB image does not have. Giving the 13-band model NDVI is part of what "having multispectral data" means, so it belongs to the comparison and is not an unfair extra. Both feature sets come from the same patches and the same training-only clipping limits from notebook 02.

**Scaling does not matter for a random forest.** The course scales features because "features with larger values/ranges can dominate the learning process" (lecture_2_introduction_to_ML.pdf, slide 49). That applies to models that combine features with weights. A tree only asks whether one feature is above a threshold, so standardising does not change its splits. We still use the clipped and standardised values so that all models start from the same preprocessing.

In [ ]:
X_rgb, X_ms, rgb_feature_names, ms_feature_names = extract_rf_features(cache, band_stats)

assert X_rgb.shape == (len(split), 6)
assert X_ms.shape == (len(split), 28)
assert np.isfinite(X_rgb).all() and np.isfinite(X_ms).all()
print("RGB features:", rgb_feature_names)
print("13-band features:", len(ms_feature_names), "columns, the last two are", ms_feature_names[-2:])

masks = {name: split["split"].eq(name).to_numpy() for name in ("train", "val", "test")}
y = split["label"].to_numpy(dtype=int)
label_ids = np.arange(len(CLASS_NAMES))
features = {"rgb": X_rgb, "13band": X_ms}

## 3. Hyperparameter tuning on the validation set only

A random forest has hyperparameters that are set before training and cannot be learned from the data (lecture_2_introduction_to_ML.pdf, slide 16). The course chooses them by hyperparameter tuning, for example a grid search (slides 28 and 30), as the last step of its workflow (slide 46). We search a small grid and score every combination on the **validation** patches only, so the test patches stay unseen until the final evaluation (lecture_2_introduction_to_ML.pdf, slide 24).

The grid covers how complex each tree may become (external: scikit-learn documentation, `RandomForestClassifier`):
- `max_depth`: `None` grows each tree until its leaves are pure; 10 and 20 stop earlier.
- `min_samples_leaf`: a split must leave at least this many training patches on each side, which smooths the model.
- `max_features`: how many features each split may choose from (`"sqrt"`, the default, or half of them).

RGB and 13 bands are tuned separately, so each input gets its own best forest and neither is held back by settings chosen for the other. We select on validation macro-F1, the same metric the project reports, because it weights every class equally. The number of trees stays at 200: more trees average more votes but do not make one tree more or less complex.

In [ ]:
PARAM_GRID = {
    "max_depth": [None, 20, 10],
    "min_samples_leaf": [1, 3],
    "max_features": ["sqrt", 0.5],
}

best_params = {}
tuning_tables = []
for input_type, X in features.items():
    table, best = tune_random_forest(X[masks["train"]], y[masks["train"]], X[masks["val"]], y[masks["val"]],
                                     PARAM_GRID, seed=0, n_estimators=N_ESTIMATORS)
    best_params[input_type] = best
    tuning_tables.append(table.assign(input=input_type))
    print(f"{input_type}: best on validation {best}")

tuning = pd.concat(tuning_tables, ignore_index=True)
tuning.to_csv(results_dir / "rf_tuning.csv", index=False)
display(tuning.groupby("input", sort=False).head(5).round(4))

## 4. Train with three seeds and save the test predictions

With the chosen settings, we train one forest per input and seed. The seed changes which training patches each tree draws and which features each split may consider (external: scikit-learn documentation, `random_state`), so the spread over seeds shows how much of a difference is just chance. We report the training scores too: a large gap between training and validation shows overfitting, a model that memorises its training data (lecture_2_introduction_to_ML.pdf, slide 20).

Every test file uses the team format from `predictions_table`: `file`, `y_true`, `y_pred`, `confidence` and the probabilities `p_0` to `p_9`. Notebook 06 reads these files for the confusion matrices and the cost-weighted errors.

In [ ]:
rows = []
models = {}
for input_type, X in features.items():
    for seed in SEEDS:
        model = RandomForestClassifier(n_estimators=N_ESTIMATORS, random_state=seed, n_jobs=-1,
                                       **best_params[input_type])
        model.fit(X[masks["train"]], y[masks["train"]])
        # predict_proba columns follow model.classes_, which must be 0..9 in order.
        assert np.array_equal(model.classes_, label_ids)
        models[(input_type, seed)] = model
        joblib.dump(model, models_dir / f"rf_{input_type}_seed{seed}.joblib")

        for subset, mask in masks.items():
            predicted = model.predict(X[mask])
            rows.append({"model": "RandomForest", "input": input_type, "seed": seed, "split": subset,
                         "accuracy": accuracy_score(y[mask], predicted),
                         "macro_f1": f1_score(y[mask], predicted, average="macro"),
                         "n_samples": int(mask.sum()), "params": str(best_params[input_type])})
            if subset == "test":
                out = predictions_table(split.loc[mask, "file"], y[mask], model.predict_proba(X[mask]))
                out.to_csv(predictions_dir / f"rf_{input_type}_seed{seed}.csv", index=False)
        print(f"Trained and saved: {input_type}, seed {seed}")

results_df = pd.DataFrame(rows)
results_df.to_csv(results_dir / "rf_results.csv", index=False)

## 5. Results: RGB versus 13 bands

Mean and standard deviation over the three seeds, for training, validation and test. The per-class reports show seed 0 only; the other seeds are in the prediction files.

In [ ]:
summary = (results_df.groupby(["input", "split"], sort=False)[["accuracy", "macro_f1"]]
           .agg(["mean", "std"]).round(4))
display(summary)

test_scores = results_df[results_df["split"].eq("test")].groupby("input")[["accuracy", "macro_f1"]].agg(["mean", "std"])
fig, ax = plt.subplots(figsize=(7, 4.5))
positions = np.arange(2)
for offset, metric in [(-0.2, "accuracy"), (0.2, "macro_f1")]:
    means = test_scores.loc[["rgb", "13band"], (metric, "mean")]
    stds = test_scores.loc[["rgb", "13band"], (metric, "std")]
    ax.bar(positions + offset, means, width=0.4, yerr=stds, capsize=4, label=metric.replace("_", "-"))
ax.set_xticks(positions, ["RGB (6 features)", "13 bands (28 features)"])
ax.set(title="Random forest on the test set (mean of 3 seeds, bars = std)", xlabel="Input", ylabel="Score",
       ylim=(0, 1))
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(figures_dir / "rf_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

for input_type in ("rgb", "13band"):
    pred = pd.read_csv(predictions_dir / f"rf_{input_type}_seed0.csv")
    print(f"\nPer-class test report: {input_type}, seed 0")
    print(classification_report(pred["y_true"], pred["y_pred"], labels=label_ids,
                                target_names=list(CLASS_NAMES), zero_division=0))

## 6. Which features does the 13-band forest use?

A random forest records how much each feature reduced the impurity of its splits, summed over all trees (external: scikit-learn documentation, `feature_importances_`, "the impurity-based feature importances"). We average this over the three seeds. It shows which bands the advantage of the 13-band input comes from. It measures how much the forest *used* a feature, not whether the feature is necessary: when two bands carry similar information, the importance can be shared between them.

In [ ]:
importance = pd.DataFrame(
    {seed: models[("13band", seed)].feature_importances_ for seed in SEEDS}, index=ms_feature_names
)
importance = importance.assign(mean=importance.mean(axis=1), std=importance.std(axis=1)).sort_values("mean")
display(importance[["mean", "std"]].sort_values("mean", ascending=False).head(10).round(4))

rgb_names = set(rgb_feature_names)
colours = ["tab:orange" if name in rgb_names else "tab:blue" for name in importance.index]
fig, ax = plt.subplots(figsize=(7, 8))
ax.barh(importance.index, importance["mean"], xerr=importance["std"], color=colours)
ax.set(title="13-band random forest: feature importance (mean of 3 seeds)",
       xlabel="Impurity-based importance", ylabel="Feature")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color="tab:orange"), plt.Rectangle((0, 0), 1, 1, color="tab:blue")],
          labels=["also in the RGB features", "only in the 13-band features"], loc="lower right")
plt.tight_layout()
plt.savefig(figures_dir / "rf_feature_importance_13band.png", dpi=150, bbox_inches="tight")
plt.show()